# 🌙 05. LunaDNA Global Terrain Fingerprint Generation

**Mission Context**: Rapid, rotation/illumination invariant topological indexing for lunar orbiter global localization.  
**Objectives**:
- Extract crater spatial geometry (64-D), directional gradient distributions (64-D), spatial pyramid texture moments (64-D), and 2D FFT radial frequency roughness (64-D).
- Concatenate into a fixed 256-D L2-normalized signature vector $\mathbf{v}_{\text{LunaDNA}} \in \mathbb{R}^{256}$.
- Analyze feature importance and cluster lunar surface regions.
- Export `lunadna_vectors.npy` and `lunadna_database.csv`.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

sys.path.append(str(Path.cwd().parent))
from lunar_core.config import load_config
from lunar_core.data_loader import LunarImageLoader
from lunar_core.lunadna import LunaDNAGenerator

config = load_config()
dna_gen = LunaDNAGenerator(vector_dim=256)


In [ ]:
# Extract LunaDNA descriptors across all dataset tiles
img_files = list(Path("data").glob("*/*.png"))
vectors = []
metadata = []

for p in img_files:
    img = LunarImageLoader.load_image(str(p))
    if img is not None:
        vec = dna_gen.generate_fingerprint(img)
        vectors.append(vec)
        metadata.append({
            "file_path": str(p),
            "file_name": p.name,
            "folder": p.parent.name
        })

vectors_arr = np.array(vectors, dtype=np.float32)
df_dna = pd.DataFrame(metadata)
print(f"Generated LunaDNA Database: {vectors_arr.shape[0]} vectors of dimension {vectors_arr.shape[1]}")


In [ ]:
# Dimensionality Reduction & Cluster Visualization (PCA)
pca = PCA(n_components=2)
coords = pca.fit_transform(vectors_arr)
df_dna['pca_1'] = coords[:, 0]
df_dna['pca_2'] = coords[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_dna, x='pca_1', y='pca_2', hue='folder', s=100, palette='viridis')
plt.title("LunaDNA 256-D Topological Descriptor Clustering (PCA Projection)", fontsize=13, fontweight='bold')
plt.xlabel(f"PC 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance)")
plt.ylabel(f"PC 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance)")
plt.grid(True, linestyle='--', alpha=0.5)

os.makedirs("outputs/visualizations", exist_ok=True)
plt.savefig("outputs/visualizations/05_lunadna_clustering.png", dpi=300)
plt.show()


In [ ]:
# Export Database: lunadna_vectors.npy & lunadna_database.csv
os.makedirs("outputs/lunadna", exist_ok=True)
np.save("outputs/lunadna/lunadna_vectors.npy", vectors_arr)
df_dna.to_csv("outputs/lunadna/lunadna_database.csv", index=False)
print("Exported outputs/lunadna/lunadna_vectors.npy and lunadna_database.csv")
